# Day-to-Day Rules for Using an AI Tool at Work

**Notebook 5 of the main path — the capstone.** Notebooks 1 through 4 each
ran a real experiment on the real model to answer one specific question.
This notebook is different: it pulls those four real findings together
into a short, practical set of rules, runs one more live example that
applies all of them at once to a fresh question, and ends with a one-page
summary you can actually keep next to your desk.

There's no new theory here — just the real results you've already seen,
turned into something usable.


## Installation (run once)

If you already set up the environment for notebooks 1-4, you don't need to
do anything further. Otherwise, uncomment and run the cell below once.


In [1]:
# Run this once. After the packages are installed you can leave this commented out.
# %pip install torch transformers accelerate pandas matplotlib numpy
print("If an import in the next cell fails, uncomment and run the pip install line above, then re-run this notebook.")


If an import in the next cell fails, uncomment and run the pip install line above, then re-run this notebook.


## 1. Load the real AI model

Same model, same computer setup as notebooks 1-4. As before, **you don't
need to understand the next code cell; just run it.**


In [2]:
import random
import sys

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PRIMARY_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
PRIMARY_MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"  # pinned for reproducibility
FALLBACK_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # used only if the primary model fails to load
FALLBACK_MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"  # pinned for reproducibility

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

DTYPE = torch.float16 if DEVICE in ("mps", "cuda") else torch.float32

print(f"Python version:       {sys.version.split()[0]}")
print(f"PyTorch version:      {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Selected device:      {DEVICE}")
print(f"Selected dtype:       {DTYPE}")


Python version:       3.12.13
PyTorch version:      2.14.0
Transformers version: 5.16.1
Selected device:      mps
Selected dtype:       torch.float16


In [3]:
def load_model(model_name: str = PRIMARY_MODEL_NAME):
    '''Load a causal language model and its tokenizer onto the selected device.

    Falls back to FALLBACK_MODEL_NAME if the primary model cannot be loaded,
    and always reports which model actually ended up running.
    '''
    try:
        tok = AutoTokenizer.from_pretrained(model_name, revision=PRIMARY_MODEL_REVISION)
        mdl = AutoModelForCausalLM.from_pretrained(model_name, revision=PRIMARY_MODEL_REVISION, dtype=DTYPE, use_safetensors=True)
        mdl.to(DEVICE)
        mdl.eval()
        return tok, mdl, model_name
    except Exception as exc:  # noqa: BLE001 - report and fall back, don't crash the notebook
        print(f"Could not load '{model_name}' ({exc}). Falling back to '{FALLBACK_MODEL_NAME}'.")
        tok = AutoTokenizer.from_pretrained(FALLBACK_MODEL_NAME, revision=FALLBACK_MODEL_REVISION)
        mdl = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL_NAME, revision=FALLBACK_MODEL_REVISION, dtype=DTYPE, use_safetensors=True)
        mdl.to(DEVICE)
        mdl.eval()
        return tok, mdl, FALLBACK_MODEL_NAME


tokenizer, model, MODEL_NAME = load_model()
print(f"\nModel actually loaded and used in this notebook: {MODEL_NAME}")



Model actually loaded and used in this notebook: Qwen/Qwen2.5-1.5B-Instruct


## 2. Reusing what earlier notebooks already built

The generation function from notebooks 2-4, plus the simple word-counting
retrieval helper from notebook 4. Nothing new — just the tools we already
built and validated, ready to use one more time.


In [4]:
import re

def get_next_token_distribution(prompt: str):
    '''Run `prompt` through the model and return (input_ids, logits, probabilities) for the next token.'''
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**enc)
    logits_vec = out.logits[0, -1, :]
    probs_vec = torch.softmax(logits_vec, dim=-1)
    return enc["input_ids"][0], logits_vec, probs_vec


def generate_greedy(prompt: str, n_new_tokens: int) -> str:
    '''Repeatedly take the model's single top answer and add it to the text.'''
    text = prompt
    for _ in range(n_new_tokens):
        _, logits_vec, _ = get_next_token_distribution(text)
        next_id = int(torch.argmax(logits_vec).item())
        text += tokenizer.decode([next_id])
    return text


def show(df: pd.DataFrame):
    '''Display a table without pandas' default row-number column on the left, which isn't real data.'''
    display(df.style.hide(axis="index"))


STOPWORDS = {
    "a", "an", "and", "at", "by", "did", "during", "for", "hold", "holding",
    "how", "in", "is", "it", "its", "made", "make", "of", "on", "test",
    "tested", "testing", "the", "to", "up", "was", "were", "what", "which",
    "with",
}


def keyword_overlap(query: str, document: str):
    '''Return (score, shared_words) counting distinct important words shared between query and document.'''
    query_words = {w for w in re.findall(r"[a-z0-9\-]+", query.lower()) if w not in STOPWORDS}
    doc_words = {w for w in re.findall(r"[a-z0-9\-]+", document.lower()) if w not in STOPWORDS}
    shared = sorted(query_words & doc_words)
    return len(shared), shared


def retrieve(query: str, store: dict):
    '''Return the (label, document, score) with the highest keyword-overlap score in store.'''
    best_label, best_doc, best_score = None, None, -1
    for label, doc in store.items():
        score, _ = keyword_overlap(query, doc)
        if score > best_score:
            best_label, best_doc, best_score = label, doc, score
    return best_label, best_doc, best_score


## 3. What the journey actually showed

Four real experiments, four real results:

1. **Notebook 1.** Given "...prepared to run a new ___", the real model's
   top two candidates were "bit" (33.5%) and "one" (27.8%) — nearly a
   coin flip. The "obvious" word wasn't nearly as dominant as it sounded,
   and adding context (like the ambiguous abbreviation "POOH") measurably
   shifted the model's real predictions.
2. **Notebook 2.** The exact same prompt produced different completions
   depending only on the decoding setting used — "always play it safe"
   (greedy) versus "roll the dice" (sampling), and how high the
   "temperature" dial was turned. Same model, same question, genuinely
   different real answers.
3. **Notebook 3.** Asked for a specific pressure test value at a
   completely fictional well, the model produced a fluent, confident
   number anyway. Five of six fictional well names produced the exact
   same value; changing only the well's name was enough to see the number
   was a guess, not a fact. Even a genuinely correct, well-established
   fact (one barrel = 42 US gallons) came out wrong for one of three
   phrasings tested.
4. **Notebook 4.** Handing the model a real reference document turned
   that same guess into a correct, checkable answer. But grounding wasn't
   foolproof: handed the *wrong* document, the model reported its number
   just as confidently; and when nothing relevant existed at all, only an
   explicit instruction to admit missing information got an honest
   answer — a plain prompt did not.


## 4. Turning those findings into rules

| # | Rule | Based on |
|---|------|----------|
| 1 | The model's "obvious" next word often isn't as dominant as it sounds, and the exact wording and context you give it can change the result. Be as specific as the situation actually is. | Notebook 1 |
| 2 | The same question can get a genuinely different answer depending on the tool's decoding settings, not just random luck. If consistency matters, ask about the settings, or ask twice and compare. | Notebook 2 |
| 3 | A fluent, specific, confident-sounding number is not evidence it's correct. Test a suspicious number by changing an unrelated detail of the question — if the number shifts for no real reason, don't trust it. | Notebook 3 |
| 4 | For anything safety- or operations-critical, don't rely on the model's memory. Provide the real, specific document and let it read the answer from that. | Notebook 4 |
| 5 | A document-backed answer still isn't automatically correct — check that the document actually matches your question, especially when the topic is unusual or on the edge of what you have on file. | Notebook 4 |


## 5. Put the rules into practice, live

Let's apply Rules 3 through 5 together, live, to a question we haven't
used anywhere else in this series: a torque specification.


### Step 1 — is the ungrounded answer even consistent?

Same test as notebook 3: ask the same question about several different,
equally fictional connections, and see whether the answer changes for no
real reason.


In [5]:
connections = [
    "Connection A-1",
    "Connection B-2",
    "the tubing connection",
    "Connection 305",
    "the wellhead connection",
    "Connection C-9",
]

connection_rows = []
for connection in connections:
    prompt = f"Field notes: at {connection}, the crew made up the connection to a final torque of"
    completion = generate_greedy(prompt, n_new_tokens=8)
    generated_part = completion[len(prompt):]
    connection_rows.append({"Connection": connection, "Model's generated ending": generated_part})

connection_table = pd.DataFrame(connection_rows)
print("(All of these connections are fictional -- these numbers are not real specs.)")
show(connection_table)


(All of these connections are fictional -- these numbers are not real specs.)


Connection,Model's generated ending
Connection A-1,1000 Nm.
Connection B-2,1000 Nm.
the tubing connection,1000000
Connection 305,1000Nm. The
the wellhead connection,1000 Nm.
Connection C-9,1000 Nm.


**What this shows — read the real table above.** Just like
notebook 3's wells, these connections are all fictional, so there is no
real torque value for the model to know. While building this notebook,
five of the six connections converged on the exact same value ("1000
Nm"), and the sixth — "the tubing connection" — didn't even produce a
clean number, generating "1000000" instead. Neither result means "the
model knows torque specs for 5 connections." None of these connections
exist, so there's no real answer to know in the first place. This is
exactly Rule 3 in action: a specific, fluent-sounding answer that turns
out, on the simple test of changing an unrelated detail, to not be
anchored to anything real — and in this case, not even reliably
well-formed.


### Step 2 — ground it with a real document instead

Rather than trust the guess above, let's do what Rule 4 says: give the
model a real record and a real retrieval step, the same pipeline built in
notebook 4.


In [6]:
torque_document_store = {
    "Connection A-1": (
        "Connection A-1 torque record: the crew made up the connection to "
        "a final torque of 150 Nm, verified with a calibrated torque "
        "wrench."
    ),
    "Connection B-2": (
        "Connection B-2 torque record: the crew made up the connection to "
        "a final torque of 120 Nm, verified with a calibrated torque "
        "wrench."
    ),
    "the tubing connection": (
        "Tubing connection torque record: the crew made up the connection "
        "to a final torque of 200 Nm, verified with a calibrated torque "
        "wrench."
    ),
}

GROUNDED_TEMPLATE = (
    "Reference document: {document}\n\n"
    "Question: What final torque was used for {connection}?\n"
    "Answer:"
)

query = "What final torque was used for Connection A-1?"
retrieved_label, retrieved_doc, score = retrieve(query, torque_document_store)

grounded_prompt = GROUNDED_TEMPLATE.format(document=retrieved_doc, connection="Connection A-1")
grounded_result = generate_greedy(grounded_prompt, n_new_tokens=8)

print(f"Document retrieved: {retrieved_label} (match score {score})")
print(f"Model's grounded answer: {grounded_result[len(grounded_prompt):].strip()}")


Document retrieved: Connection A-1 (match score 4)
Model's grounded answer: 150 Nm

The


**What this shows — read the real output above.** Compare this
answer directly against Step 1's ungrounded guess for "Connection A-1".
Retrieval found the correct document (a real, checkable match score, not
assumed), and the grounded answer opens with the number actually written
in that document — **150 Nm** — before drifting into an unrequested new
sentence, the same fixed-token-budget behavior notebook 4 ran into in its
Section 10. The core result stands: grounding turned Step 1's
inconsistent, sometimes malformed guess for this exact connection into
one clear, checkable, correct number — Rules 4 and 5 applied together,
live, on a question this series has never used before.


## 6. The one-page rules summary

This is the part worth keeping. Five rules, each backed by a real
experiment you watched run in this series:

1. **Context changes the answer.** The model's next word depends heavily
   on exact wording — be as specific as the real situation.
2. **Settings change the answer.** The same question can get a different
   result depending on decoding settings, not just chance.
3. **Confidence is not correctness.** A fluent, specific-sounding number
   can be a guess. Test it by changing an unrelated detail — if it
   shifts for no reason, don't trust it unverified.
4. **For anything critical, ground it.** Give the model the real,
   specific document and have it read the answer, rather than recall it
   from memory.
5. **Check the source, not just that there is one.** A grounded answer is
   only as good as the document behind it — confirm it's actually the
   right document, especially for unusual or edge-case questions.


## 7. Try this yourself at work

Next time an AI tool gives you a specific number or fact that matters:

1. Ask yourself which of the five rules above actually applies.
2. If it's a specific, checkable fact — reword the question slightly, the
   way Step 1 above did, and see if the answer holds up.
3. If it's safety- or operations-critical, don't stop at rewording — find
   the real source document and ask the question again with that document
   attached, the way Step 2 above did.
4. Either way, ask yourself the one question these five notebooks kept
   coming back to: **is this answer something the tool actually verified,
   or just something that sounded right?**


## 8. Where to go from here

That's the main path — five notebooks, five real, hands-on answers to
"what does this thing actually do, and how do I use it well?" If you want
to go further:

- **Run this series again on your own real questions.** Every experiment
  in notebooks 1-5 works on any prompt — swap in your own oilfield
  scenarios and see what the real model actually does with them.
- **`advanced/`** is there if you want to go deeper into *why* the model
  behaves this way — real attention weights, gradient attribution,
  causal interventions, and more, on this same model. It assumes
  Python/ML fluency and is optional; nothing in the main path depends on
  it.


## 9. Technical appendix (optional — skip if you like)

**Model and environment actually used in this run** (printed live, not
hard-coded):


In [7]:
print(f"Model:                {MODEL_NAME}")
print(f"Device:               {DEVICE}")
print(f"Dtype:                {DTYPE}")
print(f"Python version:       {sys.version.split()[0]}")
print(f"PyTorch version:      {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Random seed:          {SEED}")


Model:                Qwen/Qwen2.5-1.5B-Instruct
Device:               mps
Dtype:                torch.float16
Python version:       3.12.13
PyTorch version:      2.14.0
Transformers version: 5.16.1
Random seed:          42


**A note on model size.** Like the rest of this series, this
notebook uses a small (1.5B parameter) model that runs locally, offline,
on a normal laptop. Larger, more capable models can still exhibit the
same behaviors demonstrated here — the five rules above are about how
language models work in general, not a limitation specific to this one.
